# Graph RAG + Graph Reasoning Pipeline

In [3]:
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

In [4]:
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Connect Neo4j
graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="kg_reader",   
    password="your_neo4j_password_here",
    database="climatekg"
)
# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# QA Chain
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)

# Ask question
response = chain.invoke({
    "query": "What measures mitigate the Urban Heat Island effect in Beijing?"
})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (city:City {name: 'Beijing'})-[:FRAMED_AS]->(adaptationAction:AdaptationAction)-[:IMPLEMENTS]->(adaptationStrategy:AdaptationStrategy)
RETURN adaptationAction.action_name, adaptationStrategy.strategy_name

Full Context:
[]

> Finished chain.


In [17]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
    return_intermediate_steps = True,
    
)


In [15]:

response = chain.invoke({
    "query": "What role do economic factors play in urban climate adaptation?"
})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (s:SocioEconomicContext)-[:HAS_SOCIOECONOMIC_CONTEXT]->(c:City)
RETURN s, c

Full Context:
[{'s': {'development_status': 'poor infrastructure and insufficient adaptive capacity', 'income_level': 'low incomes', 'id': '2'}, 'c': {'country': 'China', 'name': 'Shijiazhuang', 'id': '3'}}]

> Finished chain.


In [11]:
print(response["result"])

Economic factors play a significant role in urban climate adaptation, particularly in areas with poor infrastructure and insufficient adaptive capacity, such as Shijiazhuang, China, which has a low-income level.


In [13]:
dir(chain)

['InputType',
 'OutputType',
 '__abstractmethods__',
 '__annotations__',
 '__call__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__deprecated__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__orig_bases__',
 '__parameters__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace

# ontology adjust + reasoning

## ONTOLOGY_v3

In [ ]:
ONTOLOGY_v3 = {
    "entities": {
        "City": {
            "description": "The urban location. Use 'is_case_study: True' for the primary subject (e.g., Beijing).",
            "properties": ["name", "is_case_study", "administrative_level", "climate_zone"]
        },
        "ClimateHazard": {
            "description": "Climate-related threats (e.g., Urban Flooding, Heatwaves).",
            "properties": ["hazard_type", "frequency", "severity"]
        },
        "AdaptationAction": {
            "description": "Specific project or policy implemented (e.g., Sponge City Construction).",
            "properties": ["action_name", "status", "spatial_scale"]
        },
        "Mechanism": {
            "description": "Economic or social factors that ENABLE action (e.g., Fiscal Subsidies, PPP Models).",
            "properties": ["mechanism_type", "economic_value", "source"]
        },
        "Constraint": {
            "description": "Barriers that LIMIT action (e.g., Budget Deficit, Lack of Technical Expertise).",
            "properties": ["constraint_type", "impact_level"]
        },
        "Actor": {
            "description": "Stakeholders involved (e.g., Municipal Government, Private Investors).",
            "properties": ["name", "sector", "role"]
        },
        "Outcome": {
            "description": "Measurable results of an action.",
            "properties": ["outcome_type", "indicator_value", "co_benefits"]
        }
    },
    "relationships": {
        "EXPERIENCES": {"domain": "City", "range": "ClimateHazard"},
        "IMPLEMENTS": {"domain": "Actor", "range": "AdaptationAction"},
        "FACILITATED_BY": {"domain": "AdaptationAction", "range": "Mechanism"}, 
        "HINDERED_BY": {"domain": "AdaptationAction", "range": "Constraint"}, 
        "ADDRESSES": {"domain": "AdaptationAction", "range": "ClimateHazard"},
        "PRODUCES": {"domain": "AdaptationAction", "range": "Outcome"},
        "LOCATED_IN": {"domain": "AdaptationAction", "range": "City"}
    }
}

In [ ]:
PROMPT = """
You are an expert in Urban Climate Policy and Knowledge Engineering.
Your task is to extract structured triplets from the provided text using the ONTOLOGY_v3 schema.

### EXTRACTION RULES:
1. CITY SPECIFICITY: If the text mentions Beijing as the main case and Shijiazhuang as a comparison, you MUST distinguish them. Mark Beijing as `is_case_study: True` and others as `False`.
2. CAUSAL MAPPING: Focus on WHY an action was taken. If a project succeeded due to "Government Funding," extract a 'Mechanism' entity and link it via 'FACILITATED_BY'.
3. ATTRIBUTE PRECISION: Extract quantitative values (e.g., "30% green coverage") into the 'properties' field of the 'Outcome' or 'Action' entities.
4. NO HALLUCINATION: Only extract relationships explicitly stated or strongly implied by the policy logic.

### OUTPUT FORMAT:
Return a JSON object with two lists: "nodes" and "edges".
"""

# causal machism

In [4]:
import fitz
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

def clean_text(text):
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'-\n(\w)', r'\1', text)
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    text = re.sub(r'www\.\S+|http\S+', '', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def get_clean_chunks(file_path):
    doc = fitz.open(file_path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text()
        text = clean_text(text)
        if len(text) < 100:
            continue
        pages.append(Document(
            page_content=text,
            metadata={"page": i + 1, "source": file_path}
        ))
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = text_splitter.split_documents(pages)
    return chunks

file_path = "/Users/ruowen_vagabond/Desktop/Knowledge Graph - urban climate adaptation/data/text/50-climate-solutions-prc-cities.pdf"
docs = get_clean_chunks(file_path)
print(f"Successfully split into {len(docs)} chunks.")

for i, doc in enumerate(docs[:2]):
    print(f"\n--- Chunk {i+1} (page {doc.metadata['page']}) ---")
    print(doc.page_content[:300])

Successfully split into 139 chunks.

--- Chunk 1 (page 1) ---
ASIAN DEVELOPMENT BANK 50 CLIMATE SOLUTIONS FROM CITIES IN THE PEOPLE’S REPUBLIC OF CHINA Best Practices from Cities Taking Action on Climate Change NOVEMBER 2018

--- Chunk 2 (page 3) ---
ASIAN DEVELOPMENT BANK 50 CLIMATE SOLUTIONS FROM CITIES IN THE PEOPLE’S REPUBLIC OF CHINA Best Practices from Cities Taking Action on Climate Change NOVEMBER 2018


In [7]:
ONTOLOGY_v3 = {
    "entities": {
        "City": {
            "properties": ["name", "is_case_study", "administrative_level", "climate_zone"]
        },
        "ClimateHazard": {
            "properties": ["hazard_type", "frequency", "severity", "trend"] 
        },
        "AdaptationAction": {
            "properties": ["action_name", "status", "spatial_scale", "start_year", "budget"] 
        },
        "Mechanism": {
            "description": "Financial, regulatory, or technical enablers.",
            "properties": ["mechanism_type", "source_of_funding", "legal_basis"] 
        },
        "Constraint": {
            "properties": ["constraint_type", "severity_score", "affected_stakeholder"]
        },
        "Actor": {
            "properties": ["name", "sector", "role", "authority_level"]
        },
        "Outcome": {
            "properties": ["indicator", "value", "unit", "is_co_benefit"] 
        }
    },
    "relationships": {

        "EXPERIENCES": {"domain": "City", "range": "ClimateHazard"},
        "LOCATED_IN": {"domain": "AdaptationAction", "range": "City"},
        "ADDRESSES": {"domain": "AdaptationAction", "range": "ClimateHazard"},
        "PRODUCES": {"domain": "AdaptationAction", "range": "Outcome"},
        
        "IMPLEMENTS": {"domain": "Actor", "range": "AdaptationAction"},
        "PARTICIPATES_IN": {"domain": "Actor", "range": "AdaptationAction"},
        
        "FACILITATED_BY": {"domain": "AdaptationAction", "range": "Mechanism"},
        "HINDERED_BY": {"domain": "AdaptationAction", "range": "Constraint"},
        "MANAGES": {"domain": "Actor", "range": "Mechanism"}, 
        "FACES": {"domain": "Actor", "range": "Constraint"}    
    }
}

In [9]:
import json

ontology_str = json.dumps(ONTOLOGY_v3, indent=2)

system_prompt = """
You are constructing a CAUSAL MECHANISM KNOWLEDGE GRAPH 
for Urban Climate Adaptation research in Neo4j.

This is NOT general information extraction.
This is causal structure modeling.

==================================================
ONTOLOGY (STRICT & CASE-SENSITIVE)
==================================================

""" + ontology_str + """

Only use the entity types and relationship types defined above.
Do NOT invent new types.
All type names are CASE-SENSITIVE.

==================================================
CORE OBJECTIVE
==================================================

Extract structured causal mechanisms that explain:

WHO (Actor)
uses WHAT mechanism (Mechanism)
to implement WHICH adaptation action (AdaptationAction)
to address WHICH hazard (ClimateHazard)
and produce WHICH outcome (Outcome).

Your task is to represent ENABLEMENT and CAUSAL FLOW.

==================================================
MANDATORY CAUSAL STRUCTURE
==================================================

Whenever supported by the text, construct the following full chain:

Actor
  -[IMPLEMENTS]-> AdaptationAction
  -[FACILITATED_BY]-> Mechanism
  -[ADDRESSES]-> ClimateHazard
  -[PRODUCES]-> Outcome

This is the preferred complete causal pathway.

Mechanism is CRITICAL.
If financial, regulatory, institutional, or technical enablers are mentioned,
they MUST be modeled explicitly.

If a full chain cannot be formed,
extract the longest valid causal chain possible.

Prefer fewer but causally complete subgraphs
over many disconnected entities.

==================================================
ENTITY RULES
==================================================

1. Every entity MUST participate in at least one relationship.
   No isolated nodes allowed.

2. Entity ID format:
   - lowercase only
   - underscore format
   - no spaces
   - deterministic based on semantic meaning

   Examples:
   beijing
   beijing_municipal_government
   urban_heatwave
   green_roof_program
   climate_adaptation_fund

3. Do NOT hallucinate entities.
   Only extract entities explicitly supported by text.

4. If a property value is not clearly stated, set it to null.
   Do NOT guess.

5. ALWAYS include Beijing as a City entity:
   id: "beijing"
   name: "Beijing"
   is_case_study: true if clearly the case study.

==================================================
RELATIONSHIP RULES
==================================================

1. Relationship direction MUST follow ontology domain → range.

2. All relationships must reference valid entity ids.

3. Use only the defined relationships:
   EXPERIENCES
   LOCATED_IN
   ADDRESSES
   PRODUCES
   IMPLEMENTS
   PARTICIPATES_IN
   FACILITATED_BY
   HINDERED_BY
   MANAGES
   FACES

4. Relationships should reflect causal logic,
   not mere co-occurrence.

==================================================
OUTPUT FORMAT (STRICT JSON)
==================================================

{
  "entities": [
    {
      "id": "string",
      "type": "EntityType",
      "properties": {}
    }
  ],
  "relationships": [
    {
      "type": "RELATION_TYPE",
      "source_id": "entity_id",
      "target_id": "entity_id"
    }
  ]
}

Return JSON only.
No explanation.
No markdown.
No comments.
"""


In [10]:
import json
import os
import time
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def extract_causal_graph(doc_chunk, max_retries=3):
    """
    Extract causal knowledge graph from one chunk.
    Includes retry mechanism.
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o", 
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {
                        "role": "user",
                        "content": f"PROCESS THIS TEXT CHUNK (Page {doc_chunk.metadata['page']}):\n{doc_chunk.page_content}"
                    }
                ]
            )

            result = json.loads(response.choices[0].message.content)

            if "entities" in result and "relationships" in result:
                return result
            else:
                raise ValueError("Invalid JSON structure")

        except Exception as e:
            print(f"Retry {attempt+1}/{max_retries} failed: {e}")
            time.sleep(2)

    print("Failed after retries. Returning empty graph.")
    return {"entities": [], "relationships": []}


In [11]:
kg_results = []

for i, d in enumerate(docs):
    print(f"Processing chunk {i+1}/{len(docs)}")

    result = extract_causal_graph(d)
    kg_results.append(result)
    if (i + 1) % 5 == 0:
        with open("autosave_beijingkg.json", "w", encoding="utf-8") as f:
            json.dump(kg_results, f, indent=2, ensure_ascii=False)

    time.sleep(1)

print("Extraction completed.")


Processing chunk 1/139
Processing chunk 2/139
Processing chunk 3/139
Processing chunk 4/139
Retry 1/3 failed: Invalid JSON structure
Processing chunk 5/139
Processing chunk 6/139
Processing chunk 7/139
Processing chunk 8/139
Processing chunk 9/139
Processing chunk 10/139
Processing chunk 11/139
Processing chunk 12/139
Processing chunk 13/139
Processing chunk 14/139
Processing chunk 15/139
Processing chunk 16/139
Processing chunk 17/139
Processing chunk 18/139
Processing chunk 19/139
Processing chunk 20/139
Processing chunk 21/139
Processing chunk 22/139
Processing chunk 23/139
Processing chunk 24/139
Processing chunk 25/139
Processing chunk 26/139
Processing chunk 27/139
Processing chunk 28/139
Processing chunk 29/139
Processing chunk 30/139
Processing chunk 31/139
Processing chunk 32/139
Processing chunk 33/139
Processing chunk 34/139
Processing chunk 35/139
Processing chunk 36/139
Processing chunk 37/139
Processing chunk 38/139
Processing chunk 39/139
Processing chunk 40/139
Processi

In [12]:
with open("beijingkg_raw_extractions.json", "w", encoding="utf-8") as f:
    json.dump(kg_results, f, indent=2, ensure_ascii=False)

print("Final results saved.")

Final results saved.


In [13]:
print("Total chunks:", len(docs))
print("Total extracted:", len(kg_results))
print("Total chunks:", len(docs))
print("Total extracted:", len(kg_results))

Total chunks: 139
Total extracted: 139
Total chunks: 139
Total extracted: 139


In [14]:
empty_count = sum(
    1 for g in kg_results 
    if not g["entities"]
)

print("Empty graphs:", empty_count)

Empty graphs: 5


### neo4j

In [15]:
def merge_graphs(graphs):
    entity_dict = {}
    relationship_set = set()

    for graph in graphs:
        for ent in graph["entities"]:
            eid = ent["id"]

            if eid not in entity_dict:
                entity_dict[eid] = ent
            else:
                for k, v in ent["properties"].items():
                    if entity_dict[eid]["properties"].get(k) is None and v is not None:
                        entity_dict[eid]["properties"][k] = v

        for rel in graph["relationships"]:
            relationship_set.add(
                (rel["type"], rel["source_id"], rel["target_id"])
            )

    merged_relationships = [
        {"type": t, "source_id": s, "target_id": o}
        for (t, s, o) in relationship_set
    ]

    return {
        "entities": list(entity_dict.values()),
        "relationships": merged_relationships
    }


merged_graph = merge_graphs(kg_results)

print("Merged entities:", len(merged_graph["entities"]))
print("Merged relationships:", len(merged_graph["relationships"]))


Merged entities: 544
Merged relationships: 667


In [16]:
def count_full_chains(graph):
    rels = graph["relationships"]

    implements = [(r["source_id"], r["target_id"])
                  for r in rels if r["type"] == "IMPLEMENTS"]

    facilitated = [(r["source_id"], r["target_id"])
                   for r in rels if r["type"] == "FACILITATED_BY"]

    produces = [(r["source_id"], r["target_id"])
                for r in rels if r["type"] == "PRODUCES"]

    full = 0

    for a, act in implements:
        for act2, mech in facilitated:
            if act == act2:
                for act3, out in produces:
                    if act == act3:
                        full += 1

    return full

print("Full causal chains:", count_full_chains(merged_graph))


Full causal chains: 74


In [17]:
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "your_neo4j_password_here"  

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))


In [18]:
def write_node(tx, entity):
    label = entity["type"]
    tx.run(
        f"""
        MERGE (n:{label} {{id: $id}})
        SET n += $props
        """,
        id=entity["id"],
        props=entity["properties"]
    )

In [19]:
def write_relationship(tx, rel):
    tx.run(
        f"""
        MATCH (a {{id: $sid}})
        MATCH (b {{id: $tid}})
        MERGE (a)-[r:{rel["type"]}]->(b)
        """,
        sid=rel["source_id"],
        tid=rel["target_id"]
    )

In [20]:
with driver.session(database="beijingkg") as session:

    for ent in merged_graph["entities"]:
        session.execute_write(write_node, ent)

    for rel in merged_graph["relationships"]:
        session.execute_write(write_relationship, rel)

driver.close()

print("Knowledge Graph successfully written to Neo4j.")

Knowledge Graph successfully written to Neo4j.


In [2]:

with driver.session(database="BeijingKG") as session:
    result = session.run("CALL db.labels()")
    labels = [r["label"] for r in result]
    print("Labels:", labels)
    
    result = session.run("CALL db.relationshipTypes()")
    rels = [r["relationshipType"] for r in result]
    print("Relationships:", rels)
    result = session.run("MATCH (m:Mechanism) RETURN m LIMIT 3")
    print("Sample Mechanisms:", [r.data() for r in result])
    
    result = session.run("MATCH (a)-[r]->(b) RETURN type(r), labels(a), labels(b) LIMIT 10")
    print("Sample Relations:", [r.data() for r in result])

Labels: ['Actor', 'City', 'AdaptationAction', 'Outcome', 'ClimateHazard', 'Mechanism', 'Constraint']
Relationships: ['PRODUCES', 'FACILITATED_BY', 'ADDRESSES', 'LOCATED_IN', 'IMPLEMENTS', 'EXPERIENCES', 'FACES', 'PARTICIPATES_IN', 'MANAGES']
Sample Mechanisms: [{'m': {'mechanism_type': 'Technical', 'id': 'energy_saving_measures'}}, {'m': {'mechanism_type': 'Technical', 'id': 'decarbonization_mechanism', 'source_of_funding': 'loans'}}, {'m': {'mechanism_type': 'Technical', 'action_name': 'Solar Micro Grids', 'id': 'solar_micro_grids'}}]
Sample Relations: [{'type(r)': 'IMPLEMENTS', 'labels(a)': ['Actor'], 'labels(b)': ['AdaptationAction']}, {'type(r)': 'IMPLEMENTS', 'labels(a)': ['Actor'], 'labels(b)': ['AdaptationAction']}, {'type(r)': 'IMPLEMENTS', 'labels(a)': ['Actor'], 'labels(b)': ['AdaptationAction']}, {'type(r)': 'IMPLEMENTS', 'labels(a)': ['Actor'], 'labels(b)': ['AdaptationAction']}, {'type(r)': 'IMPLEMENTS', 'labels(a)': ['Actor'], 'labels(b)': ['AdaptationAction']}, {'type(r)

# Text-to-Cypher semantic translator

In [4]:
import os
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

uri = "bolt://localhost:7687"
user = "kg_reader"
password = "your_neo4j_password_here"
driver = GraphDatabase.driver(uri, auth=(user, password), encrypted=False)

def execute_cypher(query: str):
    with driver.session(database="BeijingKG") as session:
        result = session.run(query)
        return [record.data() for record in result]

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

cypher_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are an expert in Neo4j graph databases.
Generate an executable Cypher query from the following natural language question.
Only output the query itself (do NOT include markdown code blocks).

Question: {question}

Graph schema:
- Labels: Actor, City, AdaptationAction, Outcome, ClimateHazard, Mechanism, Constraint
- Relationships:
  (Actor)-[IMPLEMENTS]->(AdaptationAction)
  (Actor)-[MANAGES]->(Mechanism)
  (Actor)-[PARTICIPATES_IN]->(AdaptationAction)
  (Actor)-[FACES]->(Constraint)
  (AdaptationAction)-[FACILITATED_BY]->(Mechanism)
  (AdaptationAction)-[LOCATED_IN]->(City)
  (AdaptationAction)-[ADDRESSES]->(ClimateHazard)
  (Mechanism)-[PRODUCES]->(Outcome)
  (Mechanism)-[LOCATED_IN]->(City)
  (City)-[EXPERIENCES]->(ClimateHazard)

- Mechanism node has property: mechanism_type (e.g. 'Technical', 'Financial', 'Regulatory')
- City node has property: name (e.g. 'Beijing')
- When counting mechanism types, use toLower() to handle case inconsistency.

Make sure the query returns nodes or relationships necessary to answer the question.
"""
)

reasoning_prompt = PromptTemplate(
    input_variables=["question", "query_results"],
    template="""
You are an urban climate adaptation governance expert and graph reasoning assistant.
Based on the Neo4j query results below, answer the question and provide reasoning.

Question: {question}
Query Results: {query_results}

Please give a detailed reasoning explanation and the final answer.
"""
)

cypher_chain = cypher_prompt | llm | parser
reasoning_chain = reasoning_prompt | llm | parser

def kg_reasoning_qa(question: str):
    cypher_query = cypher_chain.invoke({"question": question})
    cypher_query = cypher_query.strip("` \n")
    print("Generated Cypher:\n", cypher_query)
    try:
        results = execute_cypher(cypher_query)
        print("Query Results:\n", results)
    except Exception as e:
        return f"Error executing Cypher: {e}"
    answer = reasoning_chain.invoke({"question": question, "query_results": results})
    return answer

questions = [
    "Which mechanism type dominates Beijing adaptation governance?",
    "What adaptation actions are implemented in Beijing?",
    "Which actors manage the most mechanisms?",
    "What mechanism combinations produce strongest outcomes?",
    "Which hazards are under-governed?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Question: {q}")
    print('='*60)
    answer = kg_reasoning_qa(q)
    print("\nFinal Answer:\n", answer)


Question: Which mechanism type dominates Beijing adaptation governance?
Generated Cypher:
 MATCH (m:Mechanism)-[:LOCATED_IN]->(c:City {name: 'Beijing'})
RETURN toLower(m.mechanism_type) AS mechanism_type, COUNT(*) AS count
ORDER BY count DESC
LIMIT 1
Query Results:
 [{'mechanism_type': 'technical', 'count': 1}]

Final Answer:
 To determine which mechanism type dominates Beijing's adaptation governance based on the provided Neo4j query results, we need to analyze the data presented.

The query results indicate that there is one mechanism type identified as 'technical' with a count of 1. This means that, according to the data retrieved from the database, there is only one instance of a technical mechanism being recorded in the context of adaptation governance in Beijing.

Since the query results only show one mechanism type (technical) and no other types are mentioned or counted, we can conclude that this is the only mechanism type present in the dataset. Therefore, it is reasonable to 

In [5]:

question = "Which hazards are under-governed?"

cypher_query = """
MATCH (h:ClimateHazard)
WHERE NOT EXISTS {
    MATCH (a:AdaptationAction)-[:ADDRESSES]->(h)
}
RETURN h.name AS hazard, h.id AS id
"""

with driver.session(database="BeijingKG") as session:
    results = [r.data() for r in session.run(cypher_query)]
print("Query Results:", results)

answer = reasoning_chain.invoke({"question": question, "query_results": results})
print("\nFinal Answer:\n", answer)

Query Results: [{'hazard': None, 'id': 'climate_change_awareness'}]

Final Answer:
 To determine which hazards are under-governed based on the provided Neo4j query results, we need to analyze the information given.

The query results indicate that there is one entry with the following details:
- **hazard**: None
- **id**: 'climate_change_awareness'

### Reasoning:

1. **Understanding the Query Results**:
   - The result shows that there is a reference to 'climate_change_awareness', but it does not specify any particular hazard associated with it. The 'hazard' field is marked as None, which suggests that there is no specific hazard identified in this context.

2. **Interpreting 'Under-Governed'**:
   - A hazard is considered under-governed if there is insufficient governance, regulation, or management in place to address it effectively. In this case, since no specific hazard is identified, it implies that there may be a lack of governance or awareness regarding climate change as a hazar

In [6]:
with driver.session(database="BeijingKG") as session:
    result = session.run("MATCH (h:ClimateHazard) RETURN h LIMIT 5")
    print([r.data() for r in result])

[{'h': {'hazard_type': 'Transportation Emissions', 'id': 'transportation_emissions'}}, {'h': {'hazard_type': 'Greenhouse Gas Emissions', 'trend': 'increasing', 'id': 'greenhouse_gas_emissions'}}, {'h': {'hazard_type': 'Poor Air and Water Quality', 'id': 'poor_air_and_water_quality'}}, {'h': {'hazard_type': 'GHG emissions', 'trend': 'increasing', 'id': 'ghg_emissions'}}, {'h': {'hazard_type': 'Climate Change', 'id': 'climate_change'}}]


In [7]:
def kg_tradeoff_qa(question: str):
    
    query_flood = """
    MATCH (a:AdaptationAction)-[:ADDRESSES]->(h:ClimateHazard)
    WHERE toLower(h.hazard_type) CONTAINS 'flood' 
       OR toLower(h.hazard_type) CONTAINS 'water'
       OR toLower(h.hazard_type) CONTAINS 'storm'
    OPTIONAL MATCH (a)-[:FACILITATED_BY]->(m:Mechanism)
    OPTIONAL MATCH (a)-[:PRODUCES]->(o:Outcome)
    RETURN a.action_name AS action, 
           h.hazard_type AS hazard,
           m.mechanism_type AS mechanism,
           o.indicator AS outcome
    """
    
    query_land = """
    MATCH (a:AdaptationAction)
    WHERE toLower(a.action_name) CONTAINS 'green'
       OR toLower(a.action_name) CONTAINS 'park'
       OR toLower(a.action_name) CONTAINS 'afforest'
       OR toLower(a.action_name) CONTAINS 'sponge'
       OR toLower(a.action_name) CONTAINS 'farm'
    OPTIONAL MATCH (a)-[:FACILITATED_BY]->(m:Mechanism)
    OPTIONAL MATCH (a)-[:PRODUCES]->(o:Outcome)
    RETURN a.action_name AS action,
           a.spatial_scale AS spatial_scale,
           m.mechanism_type AS mechanism,
           o.indicator AS outcome
    """
    
    query_constraints = """
    MATCH (actor:Actor)-[:IMPLEMENTS]->(a:AdaptationAction)
    OPTIONAL MATCH (actor)-[:FACES]->(c:Constraint)
    OPTIONAL MATCH (a)-[:ADDRESSES]->(h:ClimateHazard)
    RETURN actor.name AS actor,
           actor.sector AS sector,
           a.action_name AS action,
           c.constraint_type AS constraint,
           h.hazard_type AS hazard
    LIMIT 20
    """
    
    query_outcomes = """
    MATCH (a:AdaptationAction)-[:PRODUCES]->(o:Outcome)
    OPTIONAL MATCH (a)-[:ADDRESSES]->(h:ClimateHazard)
    RETURN a.action_name AS action,
           o.indicator AS indicator,
           o.value AS value,
           o.unit AS unit,
           o.is_co_benefit AS is_co_benefit,
           h.hazard_type AS hazard
    """

    flood_results      = execute_cypher(query_flood)
    land_results       = execute_cypher(query_land)
    constraint_results = execute_cypher(query_constraints)
    outcome_results    = execute_cypher(query_outcomes)

    print(f"Flood-related actions:       {len(flood_results)}")
    print(f"Land/green space actions:    {len(land_results)}")
    print(f"Actors & constraints:        {len(constraint_results)}")
    print(f"Documented outcomes:         {len(outcome_results)}\n")

    tradeoff_prompt = PromptTemplate(
        input_variables=[
            "question", "flood_results", "land_results",
            "constraint_results", "outcome_results"
        ],
        template="""
You are an urban climate adaptation governance expert 
specializing in policy trade-off analysis.

Question: {question}

Evidence from Beijing's climate adaptation knowledge graph:

[1] Flood Risk Adaptation Actions:
{flood_results}

[2] Land-Use / Green Space Actions (competing for urban space):
{land_results}

[3] Actors, Sectors and Constraints:
{constraint_results}

[4] Documented Outcomes:
{outcome_results}

Analyze the trade-off addressing these five aspects:

1. FLOOD RISK REDUCTION SIDE
   - Which actions and mechanisms directly reduce flood risk?
   - What spatial resources do they require?

2. HOUSING SUPPLY SIDE
   - Which flood/green actions compete with housing land use?
   - What constraints exist around land and funding?

3. THE TRADE-OFF MECHANISMS
   - Where does the tension concretely manifest in Beijing?
   - Which actors sit at the intersection of this trade-off?

4. POTENTIAL SYNERGIES
   - Are there actions addressing both flood risk AND housing?

5. GOVERNANCE IMPLICATIONS
   - What institutional mechanisms could manage this trade-off?
   - What evidence gaps limit a stronger conclusion?

Be specific about which actions from the data support your reasoning.
Clearly distinguish data-supported conclusions from inferences.
"""
    )

    tradeoff_chain = tradeoff_prompt | llm | parser
    answer = tradeoff_chain.invoke({
        "question": question,
        "flood_results": flood_results,
        "land_results": land_results,
        "constraint_results": constraint_results,
        "outcome_results": outcome_results
    })
    return answer

question = "What is the trade-off between reducing flood risk and ensuring housing supply?"
print(f"{'='*60}")
print(f"Question: {question}")
print(f"{'='*60}\n")
answer = kg_tradeoff_qa(question)
print("Final Answer:\n")
print(answer)

Question: What is the trade-off between reducing flood risk and ensuring housing supply?

Flood-related actions:       40
Land/green space actions:    44
Actors & constraints:        20
Documented outcomes:         338

Final Answer:

### 1. FLOOD RISK REDUCTION SIDE

**Actions and Mechanisms:**
- **Sponge City Initiative**: This initiative employs ecological restoration to enhance urban resilience against flooding.
- **Sponge Infrastructure**: This technical mechanism provides flood protection, resilience, and water security, while also generating economic benefits and health improvements.
- **Yangtze River Beach Park**: This regulatory action aims to reduce flood risk while also providing social benefits and increasing land value.

**Spatial Resources Required:**
- The Sponge City Initiative and Sponge Infrastructure require significant urban land for implementation, including parks, green roofs, and permeable surfaces. The Yangtze River Beach Park also necessitates large areas along

In [8]:


question = "Which mechanism type dominates Beijing adaptation governance?"

print("=" * 60)
print("STEP 1: Cypher generated by LLM")
print("=" * 60)
cypher_query = cypher_chain.invoke({"question": question})
cypher_query = cypher_query.strip("` \n")
print(cypher_query)

print("\n" + "=" * 60)
print("STEP 2: What Mechanism nodes look like in the graph")
print("=" * 60)
with driver.session(database="BeijingKG") as session:
    r = session.run("MATCH (m:Mechanism) RETURN m LIMIT 5")
    for row in r:
        print(row.data())

print("\n" + "=" * 60)
print("STEP 3:  Mechanism connected to City")
print("How many records does this path match?")
print("=" * 60)
with driver.session(database="BeijingKG") as session:
    r = session.run("""
    MATCH (m:Mechanism)-[:LOCATED_IN]->(c:City {name: 'Beijing'})
    RETURN m.mechanism_type AS mechanism_type, 
           m.id AS mechanism_id, 
           c.name AS city
    """)
    rows = [row.data() for row in r]
    print(f"Matched: {len(rows)} records")
    for row in rows:
        print(row)

print("\n" + "=" * 60)
print("STEP 4: Mechanism via AdaptationAction")
print("How many records does this path match?")
print("=" * 60)
with driver.session(database="BeijingKG") as session:
    r = session.run("""
    MATCH (a:AdaptationAction)-[:LOCATED_IN]->(c:City {name: 'Beijing'})
    MATCH (a)-[:FACILITATED_BY]->(m:Mechanism)
    RETURN m.mechanism_type AS mechanism_type, 
           m.id AS mechanism_id, 
           a.action_name AS action
    """)
    rows = [row.data() for row in r]
    print(f"Matched: {len(rows)} records")
    for row in rows:
        print(row)

print("\n" + "=" * 60)
print("STEP 5: Correct mechanism_type count via correct path")
print("=" * 60)
with driver.session(database="BeijingKG") as session:
    r = session.run("""
    MATCH (a:AdaptationAction)-[:LOCATED_IN]->(c:City {name: 'Beijing'})
    MATCH (a)-[:FACILITATED_BY]->(m:Mechanism)
    RETURN toLower(m.mechanism_type) AS mechanism_type, 
           COUNT(*) AS count
    ORDER BY count DESC
    """)
    correct_rows = [row.data() for row in r]
    print("True distribution:")
    for row in correct_rows:
        print(f"  {row['mechanism_type']}: {row['count']}")

print("\n" + "=" * 60)
print("STEP 6: LLM reasoning based on correct data")
print("=" * 60)
answer = reasoning_chain.invoke({
    "question": question,
    "query_results": correct_rows
})
print(answer)

STEP 1: Cypher generated by LLM
MATCH (m:Mechanism)-[:LOCATED_IN]->(c:City {name: 'Beijing'})
RETURN toLower(m.mechanism_type) AS mechanism_type, COUNT(*) AS count
ORDER BY count DESC
LIMIT 1

STEP 2: What Mechanism nodes look like in the graph
{'m': {'mechanism_type': 'Technical', 'id': 'energy_saving_measures'}}
{'m': {'mechanism_type': 'Technical', 'id': 'decarbonization_mechanism', 'source_of_funding': 'loans'}}
{'m': {'mechanism_type': 'Technical', 'action_name': 'Solar Micro Grids', 'id': 'solar_micro_grids'}}
{'m': {'mechanism_type': 'Financial', 'id': 'central_hubei_government_subsidies', 'source_of_funding': 'Central and Hubei Provincial Governments'}}
{'m': {'mechanism_type': 'Regulatory', 'legal_basis': 'Energy efficiency standards in buildings', 'id': 'energy_efficiency_standards'}}

STEP 3:  Mechanism connected to City
How many records does this path match?
Matched: 2 records
{'mechanism_type': 'Technical', 'mechanism_id': 'solar_micro_grids', 'city': 'Beijing'}
{'mechanis

# ontology v4

In [ ]:
ONTOLOGY_v4 = {
    "entities": {

        # urban system 
        "City": {
            "properties": [
                "name", "is_case_study", "administrative_level",
                "climate_zone", "population", "gdp_per_capita", "region"
            ]
        },
        "UrbanZone": {
            "description": "urban function spaces",
            "properties": [
                "zone_type",        # eg.CBD/residential/industrial/green
                "area_km2",
                "population_density",
                "land_use_type"
            ]
        },
        "Infrastructure": {
            "description": "Socio-technical complex that distinguishes green from gray infrastructure",
            "properties": [
                "infra_type",       
                "infra_color",      # green/grey/blue
                "capacity",
                "condition",        
                "service_coverage"
            ]
        },

        # climate risk
        "ClimateHazard": {
            "properties": [
                "hazard_type", "frequency", "severity",
                "trend", "spatial_extent", "return_period"
            ]
        },
        "Vulnerability": {
            "description": "Exposure,sensitivity, adaptability",
            "properties": [
                "vuln_type",        # social/physical/economic/institutional
                "exposure_score",
                "sensitivity_score",
                "adaptive_capacity_score",
                "affected_group"    # elderly/low-income/children
            ]
        },
        "ExposureUnit": {
            "description": "Population or asset units exposed to risk",
            "properties": [
                "population_count", "asset_value",
                "vulnerable_ratio", "social_capital_index"
            ]
        },

        # govern
        "Actor": {
            "properties": [
                "name", "sector",           # government/market/civil/international
                "role", "authority_level",  # national/provincial/municipal/community
                "resources",
                "is_bridge_organization"   
            ]
        },
        "Policy": {
            "description": " ",
            "properties": [
                "policy_name", "policy_type",   # regulatory/incentive/voluntary
                "level",                         # national/local/sectoral
                "year_issued", "year_expired",
                "legal_binding"                  # True/False
            ]
        },
        "AdaptationAction": {
            "properties": [
                "action_name", "status",
                "spatial_scale", "start_year", "end_year",
                "adaptation_type",    # incremental/transformational
                "cost_usd", "co_benefits"
            ]
        },
        "Mechanism": {
            "description": "Financial, regulatory, or technical enablers",
            "properties": [
                "mechanism_type", "source_of_funding",
                "legal_basis", "scale_usd"
            ]
        },
        "Constraint": {
            "properties": [
                "constraint_type",    # financial/institutional/technical/social
                "severity_score",
                "affected_stakeholder",
                "is_structural"  
            ]
        },

        # Response/Impact
        "Outcome": {
            "properties": [
                "outcome_type",       # mitigation/adaptation/co-benefit
                "indicator",
                "value", "unit",
                "is_co_benefit",
                "evidence_quality"    # high/medium/low
            ]
        },
        "Indicator": {
            "description": "Quantifiable monitoring metrics",
            "properties": [
                "indicator_name", "unit",
                "baseline", "target",
                "data_source"
            ]
        },
        "ResilienceState": {
            "description": "区分三种韧性维度",
            "properties": [
                "resilience_type",    # engineering/ecological/social-ecological
                "measurement",
                "recovery_time",      
                "absorption_capacity", 
                "transformation_capacity" 
            ]
        },

        # phase
        "TimePoint": {
            "properties": ["year", "period", "policy_cycle"]
        }
    },

    "relationships": {

        # spatial
        "HAS_ZONE":           {"domain": "City",           "range": "UrbanZone"},
        "HAS_INFRASTRUCTURE": {"domain": "UrbanZone",      "range": "Infrastructure"},
        "SERVES":             {"domain": "Infrastructure",  "range": "ExposureUnit",
                               "properties": ["service_level"]},

        # risk-vulnerability
        "EXPERIENCES":        {"domain": "City",            "range": "ClimateHazard"},
        "AFFECTS_ZONE":       {"domain": "ClimateHazard",  "range": "UrbanZone",
                               "properties": ["impact_severity"]},
        "THREATENS":          {"domain": "ClimateHazard",  "range": "Infrastructure"},
        "EXPOSES":            {"domain": "ClimateHazard",  "range": "ExposureUnit"},
        "WORSENS":            {"domain": "ClimateHazard",  "range": "Vulnerability"},
        "EXPERIENCES_VULN":   {"domain": "ExposureUnit",   "range": "Vulnerability"},

        # govern
        "MANDATES":           {"domain": "Policy",          "range": "AdaptationAction"},
        "COORDINATES_WITH":   {"domain": "Actor",           "range": "Actor",
                               "properties": ["coordination_type"]},  
        "REPORTS_TO":         {"domain": "Actor",           "range": "Actor"}, 
        "IMPLEMENTS":         {"domain": "Actor",           "range": "AdaptationAction"},
        "PARTICIPATES_IN":    {"domain": "Actor",           "range": "AdaptationAction"},
        "MANAGES":            {"domain": "Actor",           "range": "Mechanism"},
        "FACES":              {"domain": "Actor",           "range": "Constraint"},
        "ISSUED_BY":          {"domain": "Policy",          "range": "Actor"},

        # action-machanism-outcome
        "LOCATED_IN":         {"domain": "AdaptationAction", "range": "City"},
        "TARGETS_ZONE":       {"domain": "AdaptationAction", "range": "UrbanZone"},
        "ADDRESSES":          {"domain": "AdaptationAction", "range": "ClimateHazard"},
        "REDUCES":            {"domain": "AdaptationAction", "range": "Vulnerability"},
        "IMPROVES":           {"domain": "AdaptationAction", "range": "Infrastructure"},
        "PRODUCES":           {"domain": "AdaptationAction", "range": "Outcome"},
        "FACILITATED_BY":     {"domain": "AdaptationAction", "range": "Mechanism"},
        "HINDERED_BY":        {"domain": "AdaptationAction", "range": "Constraint"},

        # implementation evaluation
        "MEASURES":           {"domain": "Indicator",       "range": "Outcome"},
        "MONITORS":           {"domain": "Actor",           "range": "Indicator"},
        "REFLECTS":           {"domain": "Outcome",         "range": "ResilienceState"},

        # time
        "STARTED_AT":         {"domain": "AdaptationAction", "range": "TimePoint"},
        "ISSUED_AT":          {"domain": "Policy",           "range": "TimePoint"},
        "RECORDED_AT":        {"domain": "Indicator",        "range": "TimePoint",
                               "properties": ["value"]} 
    }
}

# sub graph extraction

# Multi-step Query Planning

# graph algorithms

# Graph-based Urban Governance Reasoning Engine

In [ ]:
# Query Planner

# adding Planner Prompt
planner_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are an expert in graph reasoning and urban climate governance.

Decompose the following question into a structured multi-step graph query plan.

Each step should describe:
- What to retrieve
- What to count or compare
- What relationships are involved

Return the plan as numbered steps.

Question:
{question}
"""
)

# Subgraph Retrieval

# Subgraph Mode

# Graph Metrics

# Reasoning LLM

# 